# HuggingFace Fine-tuning Fundamentals
*Research Stage: Customizing Models for Your Data*

This notebook covers the essential concepts and practices for fine-tuning HuggingFace models on custom datasets.

## Learning Objectives
- Understand the fine-tuning process
- Prepare and preprocess custom datasets
- Implement basic fine-tuning workflows
- Evaluate and compare fine-tuned models
- Handle common fine-tuning challenges

## Prerequisites
- Basic understanding of transformers
- HuggingFace account
- GPU recommended for training
- Custom dataset (or we'll use example data)

## Setup and Environment

In [ ]:
# Install required packages (run in terminal if needed)
# pip install transformers torch datasets accelerate peft

import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling,
    DataCollatorWithPadding
)
from datasets import load_dataset, Dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Understanding Fine-tuning

Fine-tuning adapts pre-trained models to specific tasks or domains using smaller datasets.

In [ ]:
# Key concepts in fine-tuning
FINE_TUNING_CONCEPTS = {
    "transfer_learning": {
        "description": "Using knowledge from pre-trained models for new tasks",
        "benefit": "Faster training, better performance with less data"
    },
    "parameter_efficient": {
        "description": "Updating only subset of model parameters",
        "methods": ["LoRA", "QLoRA", "Adapter tuning"],
        "benefit": "Reduced memory usage, faster training"
    },
    "task_types": {
        "causal_lm": "Text generation (GPT-style)",
        "seq2seq": "Text-to-text tasks (T5, BART)",
        "classification": "Text classification",
        "token_classification": "NER, POS tagging"
    },
    "hyperparameters": {
        "learning_rate": "How much to update parameters (1e-5 to 5e-4)",
        "batch_size": "Samples per training step",
        "epochs": "Complete passes through dataset",
        "max_length": "Maximum sequence length"
    }
}

print("🎯 Fine-tuning Key Concepts:")
for concept, details in FINE_TUNING_CONCEPTS.items():
    print(f"\n{concept.upper()}:")
    if isinstance(details, dict):
        for key, value in details.items():
            print(f"  • {key}: {value}")
    else:
        print(f"  {details}")

## Dataset Preparation

Learn how to prepare and preprocess data for fine-tuning.

In [ ]:
def create_sample_dataset(task_type="generation", num_samples=100):
    """
    Create a sample dataset for demonstration.
    In practice, you'd load your own dataset.
    """
    if task_type == "generation":
        # Sample conversational data
        prompts = [
            "Hello, how are you?",
            "What's the weather like?",
            "Tell me a joke.",
            "Explain quantum physics.",
            "Write a short story about",
            "What are the benefits of",
            "How do I learn",
            "What's your favorite"
        ]
        
        responses = [
            "I'm doing well, thank you for asking! How can I help you today?",
            "I don't have access to current weather data, but I can suggest checking a weather app.",
            "Why don't scientists trust atoms? Because they make up everything!",
            "Quantum physics studies the behavior of matter and energy at atomic and subatomic scales.",
            "Once upon a time, in a digital world, there lived an AI named...",
            "Regular exercise has numerous benefits including improved cardiovascular health...",
            "Learning is a continuous process. Start with basics, practice regularly...",
            "As an AI, I don't have personal preferences, but I enjoy helping users learn!"
        ]
        
        # Create training examples
        data = []
        for i in range(num_samples):
            prompt_idx = i % len(prompts)
            response_idx = i % len(responses)
            data.append({
                "text": f"Human: {prompts[prompt_idx]}\nAssistant: {responses[response_idx]}"
            })
        
        return Dataset.from_pandas(pd.DataFrame(data))
    
    elif task_type == "classification":
        # Sample sentiment data
        texts = [
            "I love this product, it's amazing!",
            "This is terrible, I hate it.",
            "It's okay, nothing special.",
            "Best purchase I've ever made!",
            "Complete waste of money.",
            "Decent quality for the price."
        ]
        
        labels = [1, 0, 0, 1, 0, 1]  # 1 = positive, 0 = negative
        
        data = []
        for i in range(num_samples):
            text_idx = i % len(texts)
            data.append({
                "text": texts[text_idx],
                "label": labels[text_idx]
            })
        
        return Dataset.from_pandas(pd.DataFrame(data))

# Create sample datasets
print("📚 Creating sample datasets...")
gen_dataset = create_sample_dataset("generation", 50)
print(f"Generation dataset: {len(gen_dataset)} samples")

class_dataset = create_sample_dataset("classification", 50)
print(f"Classification dataset: {len(class_dataset)} samples")

# Show sample data
print("\n📝 Sample generation data:")
for i in range(3):
    print(f"  {gen_dataset[i]['text'][:100]}...")

print("\n🏷️ Sample classification data:")
for i in range(3):
    print(f"  Text: {class_dataset[i]['text']}")
    print(f"  Label: {class_dataset[i]['label']}")

## Tokenization and Preprocessing

Learn how to prepare text data for model training.

In [ ]:
def setup_tokenizer_and_model(model_name="gpt2", task_type="generation"):
    """
    Setup tokenizer and model for fine-tuning.
    """
    print(f"🔧 Setting up {model_name} for {task_type}...")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Add padding token if it doesn't exist (common for GPT models)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load model
    if task_type == "generation":
        model = AutoModelForCausalLM.from_pretrained(model_name)
    elif task_type == "classification":
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, 
            num_labels=2  # Binary classification
        )
    else:
        raise ValueError(f"Unsupported task type: {task_type}")
    
    print(f"Model parameters: {model.num_parameters():,}")
    print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")
    
    return tokenizer, model

# Setup for generation task
tokenizer, model = setup_tokenizer_and_model("gpt2", "generation")

def tokenize_function(examples, tokenizer, max_length=512):
    """
    Tokenize text data for training.
    """
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )

# Test tokenization
print("\n🔤 Testing tokenization...")
sample_text = "Hello, how are you today?"
tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.encode(sample_text)

print(f"Original: {sample_text}")
print(f"Tokens: {tokens}")
print(f"Token IDs: {token_ids}")
print(f"Decoded: {tokenizer.decode(token_ids)}")

# Tokenize dataset
print("\n📊 Tokenizing dataset...")
tokenized_dataset = gen_dataset.map(
    lambda x: tokenize_function(x, tokenizer), 
    batched=True,
    remove_columns=["text"]
)

print(f"Original dataset columns: {gen_dataset.column_names}")
print(f"Tokenized dataset columns: {tokenized_dataset.column_names}")
print(f"Sample tokenized data: {tokenized_dataset[0]}")

## Basic Fine-tuning Implementation

Implement a complete fine-tuning workflow.

In [ ]:
def fine_tune_model(model, tokenizer, dataset, output_dir="./fine_tuned_model", 
                   num_epochs=1, batch_size=4, learning_rate=5e-5):
    """
    Fine-tune a model on a dataset.
    """
    print("🚀 Starting fine-tuning...")
    print(f"Model: {model.config.name_or_path}")
    print(f"Dataset size: {len(dataset)}")
    print(f"Epochs: {num_epochs}, Batch size: {batch_size}, LR: {learning_rate}")
    
    # Split dataset
    train_test_split = dataset.train_test_split(test_size=0.2, seed=42)
    train_dataset = train_test_split["train"]
    eval_dataset = train_test_split["test"]
    
    print(f"Train samples: {len(train_dataset)}, Eval samples: {len(eval_dataset)}")
    
    # Setup data collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False  # Causal language modeling
    )
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=0.01,
        logging_steps=10,
        save_steps=50,
        evaluation_strategy="steps",
        eval_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
        report_to="none"  # Disable wandb/tensorboard logging
    )
    
    # Initialize trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
    )
    
    # Train the model
    print("\n🏃 Training in progress...")
    trainer.train()
    
    # Save the model
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    print(f"\n💾 Model saved to: {output_dir}")
    
    # Evaluate
    eval_results = trainer.evaluate()
    print(f"📊 Final evaluation results: {eval_results}")
    
    return trainer, eval_results

# Fine-tune on a small subset for demonstration
print("🎯 Fine-tuning demonstration (using small dataset)...")

# Use only a small subset for quick demo
small_dataset = tokenized_dataset.select(range(min(20, len(tokenized_dataset))))

trainer, results = fine_tune_model(
    model=model,
    tokenizer=tokenizer,
    dataset=small_dataset,
    output_dir="./demo_fine_tuned_model",
    num_epochs=1,
    batch_size=2,
    learning_rate=1e-4
)

## Testing Fine-tuned Model

Compare the fine-tuned model with the base model.

In [ ]:
def test_fine_tuned_model(base_model_name, fine_tuned_path, test_prompts):
    """
    Compare base model vs fine-tuned model performance.
    """
    print("🧪 Comparing model performance...")
    
    # Load base model
    base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
    
    # Load fine-tuned model
    ft_tokenizer = AutoTokenizer.from_pretrained(fine_tuned_path)
    ft_model = AutoModelForCausalLM.from_pretrained(fine_tuned_path)
    
    # Set pad token
    if base_tokenizer.pad_token is None:
        base_tokenizer.pad_token = base_tokenizer.eos_token
    if ft_tokenizer.pad_token is None:
        ft_tokenizer.pad_token = ft_tokenizer.eos_token
    
    for prompt in test_prompts:
        print(f"\n📝 Prompt: '{prompt}'")
        print("-" * 50)
        
        # Test base model
        inputs = base_tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = base_model.generate(
                inputs["input_ids"], 
                max_length=50, 
                num_return_sequences=1,
                pad_token_id=base_tokenizer.pad_token_id
            )
        base_response = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"🤖 Base GPT-2: {base_response[len(prompt):].strip()}")
        
        # Test fine-tuned model
        inputs = ft_tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = ft_model.generate(
                inputs["input_ids"], 
                max_length=50, 
                num_return_sequences=1,
                pad_token_id=ft_tokenizer.pad_token_id
            )
        ft_response = ft_tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"🎯 Fine-tuned: {ft_response[len(prompt):].strip()}")

# Test prompts
test_prompts = [
    "Hello, how are you",
    "Tell me about AI",
    "What is machine learning"
]

try:
    test_fine_tuned_model("gpt2", "./demo_fine_tuned_model", test_prompts)
except Exception as e:
    print(f"Testing failed: {e}")
    print("This is expected with such a small training dataset and short training time.")

## Fine-tuning Best Practices

Key lessons and recommendations for successful fine-tuning.

In [ ]:
FINE_TUNING_BEST_PRACTICES = {
    "data_quality": [
        "Use high-quality, diverse training data",
        "Ensure data is representative of target domain",
        "Clean and preprocess text appropriately",
        "Balance classes for classification tasks"
    ],
    "hyperparameter_tuning": [
        "Start with learning rates between 1e-5 and 5e-4",
        "Use smaller batch sizes on limited hardware",
        "Monitor validation loss to prevent overfitting",
        "Consider learning rate scheduling"
    ],
    "resource_management": [
        "Use mixed precision (FP16) when possible",
        "Consider gradient checkpointing for large models",
        "Use parameter-efficient methods (LoRA) for large models",
        "Monitor GPU memory usage"
    ],
    "evaluation": [
        "Always evaluate on held-out validation set",
        "Use appropriate metrics for your task",
        "Compare with baseline models",
        "Test on real-world examples"
    ],
    "common_pitfalls": [
        "Overfitting on small datasets",
        "Using too high learning rates",
        "Ignoring validation performance",
        "Not saving best model checkpoints"
    ]
}

print("💡 Fine-tuning Best Practices:")
for category, practices in FINE_TUNING_BEST_PRACTICES.items():
    print(f"\n{category.upper().replace('_', ' ')}:")
    for practice in practices:
        print(f"  ✓ {practice}")

## Advanced Fine-tuning Techniques

Brief overview of more advanced approaches.

In [ ]:
ADVANCED_TECHNIQUES = {
    "parameter_efficient_fine_tuning": {
        "description": "Update only small subset of parameters",
        "methods": {
            "LoRA": "Low-Rank Adaptation - adds trainable matrices",
            "QLoRA": "Quantized LoRA for memory efficiency",
            "Adapter_Tuning": "Adds small adapter modules between layers"
        },
        "benefits": "Reduced memory usage, faster training, prevents catastrophic forgetting"
    },
    "quantization_aware_training": {
        "description": "Train with quantization in mind",
        "approaches": ["Dynamic quantization", "Static quantization", "QAT"],
        "benefits": "Smaller model size, faster inference, lower latency"
    },
    "curriculum_learning": {
        "description": "Train on easy examples first, then harder ones",
        "implementation": "Sort data by difficulty, progressive training",
        "benefits": "Better convergence, improved final performance"
    },
    "multi_task_learning": {
        "description": "Train on multiple related tasks simultaneously",
        "benefits": "Better generalization, improved performance on all tasks"
    }
}

print("🚀 Advanced Fine-tuning Techniques:")
for technique, details in ADVANCED_TECHNIQUES.items():
    print(f"\n{technique.upper().replace('_', ' ')}:")
    print(f"  {details['description']}")
    if 'methods' in details:
        print("  Methods:")
        for method, desc in details['methods'].items():
            print(f"    • {method}: {desc}")
    if 'benefits' in details:
        print(f"  Benefits: {details['benefits']}")

## Key Takeaways and Next Steps

### What We Learned:
1. **Fine-tuning Process**: Load model → Prepare data → Tokenize → Train → Evaluate
2. **Data Preparation**: Quality and quantity matter more than model size
3. **Hyperparameters**: Learning rate and batch size are critical
4. **Evaluation**: Always validate on held-out data

### Next Steps:
- Experiment with different datasets and model sizes
- Try parameter-efficient methods (LoRA, QLoRA)
- Learn about model deployment and serving
- Explore advanced evaluation metrics

### Resources:
- [HuggingFace Fine-tuning Guide](https://huggingface.co/docs/transformers/training)
- [PEFT Library](https://huggingface.co/docs/peft/index)
- [Transformers Examples](https://github.com/huggingface/transformers/tree/main/examples)

In [ ]:
%pip install transformers datasets

from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset

# Load a pre-trained model for sequence classification
# Note: Using a small model for demo; replace with larger ones like BERT for better results
model = AutoModelForSequenceClassification.from_pretrained("microsoft/DialoGPT-small", num_labels=2)
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small")

# Load IMDB dataset (sentiment classification)
dataset = load_dataset("imdb")

# Tokenize function
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

# Apply tokenization
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,  # Short for demo
    per_device_train_batch_size=1,  # Small batch due to limited VRAM
    evaluation_strategy="epoch"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"]
)

# Train
trainer.train()

## Example: Fine-Tuning for Text Classification

Fine-tuning adapts a pre-trained model to a specific task by training on task-specific data. This example shows a basic fine-tuning for text classification using IMDB reviews.

### Steps:
1. Load model and tokenizer.
2. Prepare dataset.
3. Train with Trainer.